# Frozen financial/text analysis table and aligned MiniLM matrix

Adapted from the supplied original workflow. Outputs and personal paths have been removed. Read `docs/PREPARING_DATA.md` and configure `configs/preparation.local.json` first. Run cells in order. This adaptation has not been validated by a complete raw-to-neural rerun.


# Match datasets together

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import hashlib
import numpy as np
import pandas as pd

from pathlib import Path
import sys
REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "project_config.py").is_file()), None)
if REPO_ROOT is None:
    raise RuntimeError("Start Jupyter from the repository root or notebooks directory.")
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))
from preparation_config import load_preparation_config
PREP = load_preparation_config(REPO_ROOT)

SEC_DIR = PREP["sec_run_dir"]
FIN_DIR = PREP["financial_run_dir"]
MINILM_DIR = PREP["minilm_snapshot"].parent
OUTPUT_ROOT = PREP["analysis_output_root"]
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

BASE_CONTROLS = [
    "w_roa_current",
    "w_log_assets",
    "w_leverage",
    "w_cash_ratio",
]
FE_COLS = ["fe_year", "fe_industry_sic2"]

# Keep the baseline specification consistent with the earlier plan.
ADD_SALES_GROWTH = False
CONTROL_COLS = BASE_CONTROLS + (
    ["w_sales_growth"] if ADD_SALES_GROWTH else []
)

CURRENCY = "USD"
EXCLUDE_TEXT_REVIEW = True

EXPECTED_REPORTS = 106877
EXPECTED_CORPUS_ID = "fa8fc124c14a1ca97e34"

for folder in [SEC_DIR, FIN_DIR, MINILM_DIR]:
    assert folder.is_dir(), f"Folder not found: {folder}"

print("SEC:", SEC_DIR)
print("Compustat:", FIN_DIR)
print("MiniLM:", MINILM_DIR)

In [ ]:
def file_sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def as_bool(series):
    return (
        series.astype("string")
        .str.strip()
        .str.lower()
        .isin(["true", "1", "1.0"])
    )


def clean_cik(series):
    s = (
        series.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )
    valid = (
        s.str.fullmatch(r"\d{1,10}", na=False)
        & pd.to_numeric(s, errors="coerce").gt(0)
    )
    return s.where(valid).str.zfill(10)


# SEC data
sec_summary = read_json(SEC_DIR / "run_summary.json")
sec_path = SEC_DIR / "report_filing_dates_all.csv.gz"

assert file_sha256(sec_path) == sec_summary["report_dates_sha256"]

sec = pd.read_csv(
    sec_path,
    dtype={"doc_id": "string", "cik": "string", "accession": "string"},
    low_memory=False,
)
assert len(sec) == EXPECTED_REPORTS
assert sec["doc_id"].notna().all() and sec["doc_id"].is_unique


# Financial panel
financial_snapshot = read_json(FIN_DIR / "financial_snapshot.json")
assert financial_snapshot["complete"] is True

financial_info = financial_snapshot["files"]["financial_panel.csv.gz"]
financial_path = FIN_DIR / financial_info["name"]

assert file_sha256(financial_path) == financial_info["sha256"]

finance = pd.read_csv(
    financial_path,
    dtype={
        "gvkey": "string",
        "cik": "string",
        "comp_record_id": "string",
        "fe_year": "string",
        "fe_industry_sic2": "string",
    },
    low_memory=False,
)
assert finance["comp_record_id"].is_unique


# MiniLM report-level index and vectors
embedding_snapshot = read_json(MINILM_DIR / "latest_snapshot.json")

assert embedding_snapshot["complete"] is True
assert embedding_snapshot["corpus_id"] == EXPECTED_CORPUS_ID

index_path = MINILM_DIR / embedding_snapshot["report_index"]
matrix_path = MINILM_DIR / embedding_snapshot["report_matrix"]

assert file_sha256(index_path) == embedding_snapshot["report_index_sha256"]
assert file_sha256(matrix_path) == embedding_snapshot["report_matrix_sha256"]

text = pd.read_csv(
    index_path,
    dtype={"doc_id": "string", "cik": "string"},
    low_memory=False,
)
H_full = np.load(matrix_path, mmap_mode="r", allow_pickle=False)

assert text["doc_id"].notna().all() and text["doc_id"].is_unique
assert set(text["doc_id"]) == set(sec["doc_id"])
assert H_full.shape == (len(text), embedding_snapshot["dimension"])

positions = pd.to_numeric(text["embedding_row"], errors="raise")
assert positions.notna().all()
assert positions.eq(np.floor(positions)).all()
positions = positions.astype("int64")
assert np.array_equal(np.sort(positions), np.arange(len(text)))

text["source_embedding_row"] = positions
text = text.drop(columns="embedding_row")

for column in ["competition_intensity"]:
    text[column] = pd.to_numeric(text[column], errors="raise")

assert text["competition_intensity"].between(0, 1).all()
assert np.isfinite(H_full).all()
assert np.all(np.linalg.norm(H_full, axis=1) > 0)

sec["cik"] = clean_cik(sec["cik"])
text["cik"] = clean_cik(text["cik"])
finance["cik"] = clean_cik(finance["cik"])
finance["gvkey"] = finance["gvkey"].str.zfill(6)

for column in ["datadate", "apdedate", "effective_end",
               "next_datadate", "next_apdedate",
               "next_effective_end", "next_period_start"]:
    finance[column] = pd.to_datetime(finance[column], errors="raise")

print("SEC reports:", len(sec))
print("Financial records:", len(finance))
print("MiniLM reports:", len(text))
print("MiniLM matrix:", H_full.shape)
print("Input checks passed.")

In [ ]:
sec_columns = [
    "doc_id", "cik", "accession",
    "sec_filing_date", "sec_report_period_end", "sec_form",
    "filing_date_match_status",
]

reports = text.merge(
    sec[sec_columns].rename(columns={"cik": "sec_cik"}),
    on="doc_id",
    how="left",
    validate="one_to_one",
)

reports["cik_agrees"] = (
    reports["cik"].notna()
    & reports["sec_cik"].notna()
    & reports["cik"].eq(reports["sec_cik"])
).fillna(False)

reports["filing_date"] = pd.to_datetime(
    reports["sec_filing_date"], errors="coerce"
)
reports["report_period_end"] = pd.to_datetime(
    reports["sec_report_period_end"], errors="coerce"
)

reports["sec_dates_usable"] = (
    reports["filing_date_match_status"].eq("matched_cik_and_accession")
    & reports["cik_agrees"]
    & reports["filing_date"].notna()
    & reports["report_period_end"].notna()
    & reports["filing_date"].ge(reports["report_period_end"])
    & reports["sec_form"].eq("10-K")
)

links = finance[["cik", "gvkey"]].dropna().drop_duplicates()

candidate_counts = (
    links.groupby("cik")["gvkey"]
    .nunique()
    .rename("n_candidate_gvkeys")
    .reset_index()
)

unique_links = links.loc[
    links["cik"].isin(
        candidate_counts.loc[
            candidate_counts["n_candidate_gvkeys"].eq(1), "cik"
        ]
    )
]

reports = (
    reports.merge(
        candidate_counts, on="cik", how="left", validate="many_to_one"
    )
    .merge(
        unique_links, on="cik", how="left", validate="many_to_one"
    )
)

reports["company_link_status"] = np.select(
    [
        reports["cik"].isna(),
        reports["n_candidate_gvkeys"].isna(),
        reports["n_candidate_gvkeys"].gt(1),
    ],
    [
        "missing_cik",
        "no_compustat_candidate",
        "multiple_gvkeys_review",
    ],
    default="single_gvkey_candidate",
)

reports["historical_identity_verified"] = False

print("SEC dates usable:", int(reports["sec_dates_usable"].sum()))
print("\nCompany candidate status:")
display(reports["company_link_status"].value_counts())

In [ ]:
financial_columns = [
    "gvkey", "comp_record_id", "conm", "fyear",
    "datadate", "apdedate", "effective_end", "period_end_source",
    "curcd", "roa", "roa_status",
    *BASE_CONTROLS, *FE_COLS,
    "w_sales_growth", "sic2",
    "multiple_periods_in_fyear", "duplicate_effective_period",
    "period_end_label_review", "current_period_standard_gap",
    "next_comp_record_id", "next_fyear",
    "next_datadate", "next_apdedate", "next_effective_end",
    "next_period_start", "next_actual_gap_days",
    "next_roa_candidate_standard_gap",
    "next_duplicate_effective_period",
    "roa_next_fyear", "next_outcome_status",
]

periods = finance[financial_columns].rename(
    columns={
        "conm": "comp_company_name",
        "fyear": "comp_fyear",
        "next_fyear": "next_comp_fyear",
    }
)

eligible_identifiers = reports.loc[
    reports["sec_dates_usable"] & reports["gvkey"].notna(),
    ["doc_id", "gvkey", "report_period_end"],
]

period_candidates = eligible_identifiers.merge(
    periods,
    left_on=["gvkey", "report_period_end"],
    right_on=["gvkey", "effective_end"],
    how="inner",
    validate="many_to_many",
)

period_counts = (
    period_candidates.groupby("doc_id")
    .size()
    .rename("n_exact_period_candidates")
    .reset_index()
)

unique_periods = period_candidates.loc[
    ~period_candidates["doc_id"].duplicated(keep=False)
].drop(columns=["gvkey", "report_period_end"])

reports = (
    reports.merge(
        period_counts, on="doc_id", how="left", validate="one_to_one"
    )
    .merge(
        unique_periods, on="doc_id", how="left", validate="one_to_one"
    )
)

reports["period_link_status"] = np.select(
    [
        ~reports["sec_dates_usable"],
        reports["gvkey"].isna(),
        reports["n_exact_period_candidates"].isna(),
        reports["n_exact_period_candidates"].gt(1),
    ],
    [
        "sec_dates_or_identity_unresolved",
        "company_candidate_unresolved",
        "no_exact_financial_period",
        "multiple_exact_periods_review",
    ],
    default="unique_exact_period_candidate",
)

# Flag every repeated company-period report; do not select based on outcomes.
linked = reports["period_link_status"].eq("unique_exact_period_candidate")
reports["duplicate_company_period"] = False
reports.loc[linked, "duplicate_company_period"] = (
    reports.loc[linked]
    .duplicated(["gvkey", "comp_record_id"], keep=False)
    .to_numpy()
)

print("Financial period matching:")
display(reports["period_link_status"].value_counts())

print(
    "Reports involved in repeated company periods:",
    int(reports["duplicate_company_period"].sum()),
)

In [ ]:
for column in CONTROL_COLS + ["roa_next_fyear"]:
    reports[column] = pd.to_numeric(reports[column], errors="coerce")

reports["controls_complete"] = np.isfinite(
    reports[CONTROL_COLS].to_numpy(dtype=float)
).all(axis=1)

reports["fixed_effect_fields_complete"] = (
    reports[FE_COLS].notna().all(axis=1)
)

reports["outcome_available"] = (
    reports["next_outcome_status"].eq("next_fyear_roa_candidate")
    & np.isfinite(reports["roa_next_fyear"])
)

reports["standard_next_period"] = as_bool(
    reports["next_roa_candidate_standard_gap"]
)

reports["financial_period_review"] = (
    as_bool(reports["multiple_periods_in_fyear"])
    | as_bool(reports["duplicate_effective_period"])
    | as_bool(reports["next_duplicate_effective_period"])
    | as_bool(reports["period_end_label_review"])
)

reports["text_review_flag"] = as_bool(reports["review_required"])

# Timing diagnostics, not a new outcome definition.
next_start = pd.to_datetime(reports["next_period_start"], errors="coerce")
next_end = pd.to_datetime(reports["next_effective_end"], errors="coerce")

timing_known = (
    reports["filing_date"].notna()
    & next_start.notna()
    & next_end.notna()
    & next_end.ge(next_start)
)

reports["next_period_fully_after_filing"] = pd.Series(
    pd.NA, index=reports.index, dtype="boolean"
)
reports.loc[timing_known, "next_period_fully_after_filing"] = (
    next_start.loc[timing_known]
    .gt(reports.loc[timing_known, "filing_date"])
)

reports["next_period_days_before_filing"] = np.nan

# Count accounting-period days strictly before the filing date.
overlap_end = pd.concat(
    [
        reports["filing_date"] - pd.Timedelta(days=1),
        next_end,
    ],
    axis=1,
).min(axis=1)

reports.loc[timing_known, "next_period_days_before_filing"] = (
    (overlap_end - next_start).dt.days.add(1).clip(lower=0)
    .loc[timing_known]
)

reports["outcome_ends_after_filing"] = (
    timing_known & next_end.gt(reports["filing_date"])
)

# Separate reasons: a report can fail more than one check.
flags = pd.DataFrame({
    "financial_period_unresolved":
        reports["period_link_status"].ne("unique_exact_period_candidate"),
    "duplicate_company_period":
        reports["duplicate_company_period"],
    "financial_period_review":
        reports["financial_period_review"],
    "controls_missing":
        ~reports["controls_complete"],
    "year_or_industry_missing":
        ~reports["fixed_effect_fields_complete"],
    "future_roa_unavailable":
        ~reports["outcome_available"],
    "next_period_length_review":
        ~reports["standard_next_period"],
    "outcome_not_ending_after_filing":
        ~reports["outcome_ends_after_filing"],
    "outside_currency_scope":
        reports["curcd"].ne(CURRENCY).fillna(True),
    "text_review_required":
        reports["text_review_flag"] & EXCLUDE_TEXT_REVIEW,
}).fillna(True).astype(bool)

reports["exclusion_reasons"] = flags.apply(
    lambda row: ";".join(row.index[row.to_numpy(dtype=bool)]),
    axis=1,
)

reports["eligible_provisional"] = ~flags.any(axis=1)
reports["extreme_future_roa"] = reports["roa_next_fyear"].abs().gt(1)

print("Reports passing provisional data screens:",
      int(reports["eligible_provisional"].sum()))

print("\nExclusion counts — reasons can overlap:")
display(flags.sum().sort_values(ascending=False).to_frame("reports"))

print("\nTiming among provisional candidates:")
display(
    reports.loc[
        reports["eligible_provisional"],
        "next_period_fully_after_filing",
    ].value_counts(dropna=False)
)

print("\nImportant: unique company candidates remain historically unverified.")
print("This is provisional data readiness, not causal identification.")

In [ ]:
RUN_DIR = OUTPUT_ROOT / (
    datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S")
)
RUN_DIR.mkdir(parents=True, exist_ok=False)

ready = (
    reports.loc[reports["eligible_provisional"]]
    .sort_values("doc_id")
    .reset_index(drop=True)
)

assert ready["doc_id"].is_unique
assert not ready.duplicated(["gvkey", "comp_record_id"]).any()

source_rows = ready["source_embedding_row"].to_numpy(dtype="int64")
H = np.asarray(H_full[source_rows], dtype="float32")

assert H.shape == (len(ready), H_full.shape[1])
assert np.isfinite(H).all()

ready["embedding_row"] = np.arange(len(ready))
ready["status"] = "provisional_full_sample"

TABLE_NAME = "geni_report_complete_with_controls.csv"
MATRIX_NAME = "geni_report_embeddings_complete.npy"

ready.to_csv(RUN_DIR / TABLE_NAME, index=False)
np.save(RUN_DIR / MATRIX_NAME, H, allow_pickle=False)

# Preserve every report, including unresolved and excluded ones.
reports.to_csv(
    RUN_DIR / "report_alignment_audit.csv.gz",
    index=False,
    compression="gzip",
)

reports.loc[~reports["eligible_provisional"]].to_csv(
    RUN_DIR / "reports_requiring_review.csv.gz",
    index=False,
    compression="gzip",
)

period_candidates.to_csv(
    RUN_DIR / "exact_period_candidates.csv.gz",
    index=False,
    compression="gzip",
)

summary = {
    "all_reports": len(reports),
    "sec_dates_usable": int(reports["sec_dates_usable"].sum()),
    "unique_financial_period_matches": int(
        reports["period_link_status"]
        .eq("unique_exact_period_candidate").sum()
    ),
    "provisional_reports": len(ready),
    "provisional_companies": int(ready["gvkey"].nunique()),
    "dimension": int(H.shape[1]),
    "currency": CURRENCY,
    "numeric_controls": CONTROL_COLS,
    "categorical_controls": FE_COLS,
    "exclusion_counts": {
        k: int(v) for k, v in flags.sum().items()
    },
    "sec_input": str(sec_path),
    "financial_input": str(financial_path),
    "embedding_snapshot": str(MINILM_DIR / "latest_snapshot.json"),
}

snapshot = {
    **summary,
    "complete": True,
    "table": TABLE_NAME,
    "matrix": MATRIX_NAME,
    "table_sha256": file_sha256(RUN_DIR / TABLE_NAME),
    "matrix_sha256": file_sha256(RUN_DIR / MATRIX_NAME),
    "corpus_id": EXPECTED_CORPUS_ID,
    "historical_identity_verified": False,
    "outcome": "next fiscal-year ROA; may overlap pre-filing months",
    "note": (
        "Year/industry categories are prepared, not yet fitted. "
        "No causal model has been estimated."
    ),
}

(RUN_DIR / "dataset_snapshot.json").write_text(
    json.dumps(snapshot, indent=2), encoding="utf-8"
)

# Verify the exported CSV/vector alignment.
check = pd.read_csv(
    RUN_DIR / TABLE_NAME,
    dtype={"doc_id": "string"},
)
assert check["doc_id"].tolist() == ready["doc_id"].tolist()
assert np.array_equal(check["embedding_row"], np.arange(len(ready)))
assert np.load(
    RUN_DIR / MATRIX_NAME, mmap_mode="r"
).shape == (len(ready), H.shape[1])

print(json.dumps(summary, indent=2))
print("\nMATCHING AND EXPORT COMPLETE")
print("GENI input folder:", RUN_DIR)
print("Table:", RUN_DIR / TABLE_NAME)
print("Matrix:", RUN_DIR / MATRIX_NAME)